In [ ]:
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help

In [ ]:
import logging
import warnings
warnings.filterwarnings('ignore')

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

注意：运行下面的代码首先要执行 [gen_demo_factor_data.py](../tools/gen_demo_factor_data.py) 脚本生成示例数据。

# HDF5DB

`HDF5DB` 是基于本地 [HDF5 文件](https://www.hdfgroup.org/) 构建的因子库，继承自 `WritableFactorDB`，因此支持完整的读写和变更操作。

## 存储结构

HDF5DB 的逻辑层与存储层的对应关系：

```
MainDir/                     ← 因子库
  ├── table_1/               ← 因子表（一个子目录）
  │     ├── factor_a.hdf5    ← 因子（一个 HDF5 文件）
  │     ├── factor_b.hdf5
  │     ├── _TableInfo.h5    ← 表的元信息
  │     └── _Table.lock      ← 表锁
  ├── table_2/
  │     └── ...
  └── _FDB.lock              ← 库锁
```

每个 HDF5 因子文件内部包含三个 Dataset：

| Dataset | 内容 | 说明 |
|---------|------|------|
| `ID` | 证券代码序列 | shape=(n_ids,), dtype=String, utf-8 编码 |
| `DateTime` | 时点序列 | shape=(n_dts,), dtype=float, 时点转为 timestamp 存储 |
| `Data` | 因子数据矩阵 | shape=(n_dts, n_ids), double 存为 float64, string 存为 String, object 存为 vlen_dtype(uint8) |

因子的元数据存储在 HDF5 文件 root group 的 attrs 中，其中必须包含 `DataType` 属性。

### 锁机制

HDF5DB 使用文件锁保证并发安全：
- **库锁**（`_FDB.lock`）：修改因子库结构（创建/重命名/删除表）时需要获取
- **表锁**（`_Table.lock`）：修改因子表结构（重命名/删除因子）或读写元信息时需要获取
- **因子锁**（`<factor_name>.lock`）：读写因子数据或元信息时需要获取

```mermaid
graph TD
    subgraph 逻辑层
        A[因子库<br/>HDF5DB]
        B1[因子表<br/>HDF5FactorTable]
        B2[因子表]
        A --> B1
        A --> B2
        F1[因子<br/>Factor]
        F2[因子]
        B1 --> F1
        B1 --> F2
    end

    subgraph 存储层
        C[主目录<br/>MainDir]
        D1[子目录<br/>table_1]
        D2[子目录<br/>table_2]
        C --> D1
        C --> D2
        E1[HDF5文件<br/>factor_a.hdf5]
        E2[HDF5文件<br/>factor_b.hdf5]
        D2 --> E1
        D2 --> E2
    end

    subgraph HDF5内部
        DS1[DateTime Dataset]
        DS2[ID Dataset]
        DS3[Data Dataset]
    end

    A -.-> C
    B1 -.-> D2
    F1 -.-> E2
    E2 -.-> HDF5内部

    style A fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style C fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style DS1 fill:#ff8c00,color:#fff,stroke:#cc7000,stroke-width:2px
    style DS2 fill:#ff8c00,color:#fff,stroke:#cc7000,stroke-width:2px
    style DS3 fill:#ff8c00,color:#fff,stroke:#cc7000,stroke-width:2px
```

# 创建与连接

HDF5DB 的核心构造参数：

| 参数 | 类型 | 说明 |
|------|------|------|
| `MainDir` | `pathlib.Path` | **必填**，存放数据的主目录路径 |
| `FileOpenRetryNum` | `int \| inf` | 打开文件失败时的重试次数，默认 `inf`（无限重试） |

默认配置文件路径为 `~/QuantStudioConfig/HDF5DBConfig.json`。

In [ ]:
# 创建因子库对象并 connect
from QuantStudio.Factor.HDF5DB import HDF5DB

FDB = HDF5DB(args={"MainDir": "../data/HDF5"}).connect()
print(qs_help(FDB))

In [ ]:
# 获取因子库中的因子表列表
print(FDB.TableNames)

# 因子表

`HDF5FactorTable` 继承自 `FactorTable`，在其基础上增加了 HDF5 特有的参数：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `LookBack` | `int \| inf` | `0` | 缺失填充最大回溯天数。`0` 表示不填充，`inf` 表示无限制回溯填充 |
| `OnlyStartLookBack` | `bool` | `False` | 为 `True` 时只对提取数据的第一个时点进行缺失填充 |
| `OnlyLookBackNontarget` | `bool` | `False` | 为 `True` 时只使用不在目标时点序列中的数据来填充缺失 |
| `OnlyLookBackDT` | `bool` | `False` | 为 `True` 时所有 ID 统一沿时点维度回溯填充，不按每个 ID 单独填充 |
| `TargetDT` | `datetime \| None` | `None` | 非 `None` 时忽略时点序列，只取该时点的值广播到所有请求时点 |

In [ ]:
# 获取因子表对象
FT = FDB.getTable("stock_cn_day_bar", args={"LookBack": 0})
print(qs_help(FT))

In [ ]:
# 获取因子表中的因子列表
print(FT.FactorNames)

## 缺失填充

`LookBack` 参数控制缺失值的回溯填充行为。HDF5 文件的优势在于可以高效地读取历史数据来做缺失填充。

In [ ]:
# LookBack=0：不填充缺失
FT = FDB.getTable("stock_cn_day_bar", args={"LookBack": 0})

DTs = [dt.datetime(2025, 1, 3), dt.datetime(2025, 1, 4), dt.datetime(2025, 1, 5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = FT.readData(factor_names=["open", "close"], ids=IDs, dts=DTs).loc[:, DTs, IDs]
print("LookBack=0 :")
print(Data)

print("-" * 10)
# LookBack=2：最多回溯 2 天填充缺失
FT2 = FDB.getTable("stock_cn_day_bar", args={"LookBack": 2})
Data2 = FT2.readData(factor_names=["open", "close"], ids=IDs, dts=DTs).loc[:, DTs, IDs]
print("LookBack=2 :")
print(Data2)

## 元信息

In [ ]:
# 获取因子表元信息
print(FT.getMetaData())

In [ ]:
# 获取因子元信息（批量）
print(FT.getFactorMetaData(factor_names=["close", "open"]))

## 时点序列与 ID 序列

HDF5FactorTable 的 `getDateTime` 和 `getID` 方法直接从 HDF5 文件的 Dataset 中读取。

传入 `ifactor_name` 可以获取特定因子的时点/ID；传入 `idt`（对 getID）或 `iid`（对 getDateTime）可以进一步过滤到有数据的范围。

In [ ]:
# 获取时点序列
print(FT.getDateTime(start_dt=dt.datetime(2025, 3, 1), end_dt=dt.datetime(2025, 3, 5)))

In [ ]:
# 获取 ID 序列
IDs = FT.getID()
print(IDs[0], ", ..., ", IDs[-1], f"共 {len(IDs)} 个")

## 读取数据

数据的读取有两种方式：
- **因子表级别 `readData`**：返回 `Panel`（继承自 FactorTable），适用于批量读取多个因子
- **单因子级别 `readFactorData`**：返回 `DataFrame`（HDF5FactorTable 特有），直接读取单个 HDF5 文件，更高效

In [ ]:
# 因子表级别 readData：返回 Panel
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = FT.readData(factor_names=["open", "close"], ids=IDs, dts=DTs)
print("因子表数据 (Panel):")
print(Data)

In [ ]:
# 单因子级别 readFactorData：直接读 HDF5 文件
print(qs_help(FT.readFactorData))

In [ ]:
# 使用 readFactorData 读取单个因子的原始数据
Data = FT.readFactorData(ifactor_name="close", ids=IDs, dts=DTs)
print("单因子数据 (DataFrame):")
print(Data)

# 因子

通过 `FT.getFactor()` 获取的因子对象是通用的 `Factor` 实例，其 `readData`、`getID`、`getDateTime` 等 API 与其他因子库中的因子完全一致。详细说明请参见 **[基本框架](基本框架.ipynb)** 和 **[因子开发](因子开发.ipynb)**。

In [ ]:
# 获取因子对象
F = FT.getFactor("close")
print(F.getMetaData(key=None))

In [ ]:
# 因子读取数据
Data = F.readData(ids=["000001.SZ", "000002.SZ"], dts=DTs)
print(Data)

# 因子库的变更操作

HDF5DB 继承自 `WritableFactorDB`，支持完整的写入和变更操作。以下操作均受锁机制保护。

## 数据写入

HDF5DB 提供了两个写入方法：

| 方法 | 输入 | 说明 |
|------|------|------|
| `writeData` | `Panel` | 批量写入多个因子，内部调用 `writeFactorData` |
| `writeFactorData` | `DataFrame` | 写入单个因子的数据，支持增量更新 |

两者均支持 `if_exists` 参数控制写入冲突处理：

| `if_exists` | 说明 |
|-------------|------|
| `"update"` | 用新数据更新已存在的值（默认） |
| `"replace"` | 完全用新数据替换，相当于先删旧数据再写入 |
| `"append"` | 只写入原来没有的（NaN 的）位置，不影响已有数据 |
| `"update_notnull"` | 只更新原来有值的位置（新数据非空才更新） |

In [ ]:
# 创建测试数据
from QuantStudio.Core.QSObject import Panel

IDs = [str(i).zfill(6) + ".SZ" for i in range(1, 4)]
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]

Data = Panel({
    "Factor1": pd.DataFrame(np.random.randn(5, 3), index=DTs, columns=IDs),
    "Factor2": pd.DataFrame(np.random.randn(5, 3), index=DTs, columns=IDs)
})
print("待写入的数据 : ")
print(Data)

In [ ]:
# 使用 writeData 批量写入
TargetTable = "TestTable"
FDB.writeData(data=Data, table_name=TargetTable, if_exists="update", data_type={"Factor1":"double", "Factor2":"double"})

print("写入后的因子表列表 : ")
print(FDB.TableNames)
print()
print("写入后的因子列表 : ")
print(FDB.getTable(TargetTable).FactorNames)

In [ ]:
# 使用 writeFactorData 追加单个因子
NewFactor = pd.DataFrame(np.random.randn(5, 3), index=DTs, columns=IDs)
FDB.writeFactorData(factor_data=NewFactor, table_name=TargetTable, ifactor_name="Factor3", if_exists="update", data_type="double")

print("追加后的因子列表 : ")
print(FDB.getTable(TargetTable).FactorNames)

## 元信息管理

In [ ]:
# 设置因子的元信息（支持单个 key 或批量 meta_data）
FDB.setFactorMetaData(table_name=TargetTable, ifactor_name="Factor1", key="Description", value="这是一个测试因子")

FT = FDB.getTable(TargetTable)
print("设置后的元信息 : ")
print(FT.getFactorMetaData(factor_names=["Factor1"]))

In [ ]:
# 设置因子表的元信息
FDB.setTableMetaData(table_name=TargetTable, meta_data={"Description": "这是一张测试表", "Version": 1})

print("设置后的表元信息 : ")
print(FT.getMetaData())

## 重命名与删除

In [ ]:
# 重命名因子
print("重命名前 : ", FT.FactorNames)
FDB.renameFactor(table_name=TargetTable, old_factor_name="Factor1", new_factor_name="NewFactor1")
print("重命名后 : ", FT.FactorNames)

In [ ]:
# 删除因子
FDB.deleteFactor(table_name=TargetTable, factor_names=["NewFactor1"])
print("删除后 : ", FT.FactorNames)

In [ ]:
# 重命名因子表
print("重命名前 : ", FDB.TableNames)
FDB.renameTable(old_table_name=TargetTable, new_table_name="NewTestTable")
print("重命名后 : ", FDB.TableNames)

In [ ]:
# 删除因子表
FDB.deleteTable(table_name="NewTestTable")
print("删除后 : ", FDB.TableNames)

# 数据维护

HDF5DB 提供了两个数据维护工具方法：

| 方法 | 说明 |
|------|------|
| `optimizeData(table_name, factor_names)` | 对指定因子按 DateTime 排序以优化存储。当大量追加写入导致数据乱序后，可调用此方法整理 |
| `fixData(table_name, factor_names)` | 修复数据不一致问题：ID/Datetime 长度与 Data 不匹配、ID/Datetime 重复值等 |